In [1]:
print("ALLOKAY")

ALLOKAY


In [ ]:
## Pre requisites:
## Tested with COLLAB

In [7]:
import os
import bs4

import langchain
from langchain import hub

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

from langchain_groq import ChatGroq

from langchain_core.output_parsers import StrOutputParser
from langchain.text_splitter import RecursiveCharacterTextSplitter

## Load ENV File

In [4]:
from dotenv import load_dotenv,find_dotenv

In [5]:
load_dotenv(find_dotenv())

True

In [6]:
## Load from .env file
os.getenv("GROQ_API_KEY")

''

In [ ]:
os.environ["GROQ_API_KEY"] =  os.getenv("GROQ_API_KEY")

## Collection of data

In [ ]:
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        #filter specific parts of the webpage, improving efficiency.
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

In [ ]:
docs=loader.load()

In [ ]:
docs[0].metadata

In [ ]:
print(docs[0].page_content)

## **Create Embedding Model**

In [ ]:
## Embedding is done with Transformer models. We will use HuggingFaceBgeEmbeddings for this purpose. It is a wrapper around the BGE model from Hugging Face.

In [ ]:
## this is Embedding model
model_name="BAAI/bge-small-en"

In [ ]:
encode_kwargs={"normalize_embeddings": True}

In [8]:
model_kwargs={"device": "cpu"}

In [ ]:
## EMBEDDING MODEL ===> HuggingFaceBgeEmbeddings


hf_embeddings=HuggingFaceBgeEmbeddings(
    model_name=model_name, 
    model_kwargs=model_kwargs, 
    encode_kwargs=encode_kwargs
)

## **Chunking the data**

In [ ]:
## Number of Words in the document
len(docs[0].page_content)

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)

In [ ]:
len(splits)

In [ ]:
splits = text_splitter.split_documents(docs)

## Store the data in Vector Database

In [ ]:
# # sotre all data in vector database
# vectorstore = FAISS.from_documents(
#     documents=docs,
#     embedding=hf_embeddings
# )

vectorstore = FAISS.from_documents(
    documents=splits,
    embedding=hf_embeddings
)

## Create Retriever

In [ ]:
### Create retriever from vectorstore
retriever=vectorstore.as_retriever()

## Create Large Language Model

In [ ]:
import pprint
from langchain_core.runnables import RunnablePassthrough

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
## Language Model ===> ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant")
# llama3-8b-8192 --> Decommisioned
# llm=ChatGroq(model="llama3-8b-8192")

In [ ]:
# Import the prompt template from the hugging face hub
prompt = hub.pull("rlm/rag-prompt")

In [ ]:
prompt.messages

In [ ]:
pprint.pprint(prompt.messages)

In [ ]:
prompt

In [ ]:
# RunnablePassthrough is used to pass the retrieved documents to the prompt template without any modification.

In [ ]:
# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain.invoke("What is Task Decomposition?")